In [ ]:
import numpy as np
import joblib
from utils import train_categorical_model, prepare_datasets, validate_results, train_incident_agent, generate_oof_features, find_optimal_threshold
from rcf_model import RCF
from meta_scorer import train_fusion_meta_learner, CostSensitiveMetaLearner, _incident_entropy
from catboost import CatBoostClassifier
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
file_path = r"UNSW-NB15\unsw_train.csv"

# FIX (PCA Leakage): prepare_datasets now returns X_num_raw (6th value) —
# the raw, unscaled numerical frame. This is passed to generate_oof_features
# so each fold fits its own scaler+PCA exclusively on its training split.
# X_train_num (PCA-transformed using the full-train artifact) is kept for
# the final RCF training step in cell 5.
X_train_cat, X_train_num, X_train_num_raw, y_bin, cat_cols, y_multi = prepare_datasets(file_path, is_train=True)

# FIX (PCA Leakage): Pass X_train_num_raw instead of X_train_num.
# generate_oof_features will fit a fresh scaler+PCA per fold internally.
oof_cat_scores, oof_rcf_scores, oof_incident_entropy = generate_oof_features(
    X_train_cat, X_train_num_raw, y_bin, cat_cols,
    train_cat_func=train_categorical_model,
    rcf_class=RCF,
    train_incident_func=train_incident_agent,
    n_splits=5,
    y_multi=y_multi
)

In [ ]:
X_meta_train_clean = np.column_stack((oof_cat_scores, oof_rcf_scores, oof_incident_entropy))

COST_FN = 10
COST_FP = 2

meta_learner = train_fusion_meta_learner(
    X_meta_train_clean,
    y_bin,
    COST_FN=COST_FN,
    COST_FP=COST_FP,
    lambda_reg=0.1
)

meta_learner.save_model(r"Saves/meta_learner.pkl")

# Compute the optimal decision threshold from OOF scores.
# Done here — on OOF predictions — so the threshold is chosen
# without any information from the held-out test set.
# The result is saved to Saves/optimal_threshold.json so the
# SOAR agent can load it at runtime without re-running training.
oof_meta_scores = meta_learner.predict_proba(X_meta_train_clean)
optimal_threshold = find_optimal_threshold(
    y_bin,
    oof_meta_scores,
    cost_fn=COST_FN,
    cost_fp=COST_FP,
    save_path="Saves/optimal_threshold.json"
)

In [ ]:
# Train full-dataset base models (no OOF needed here — these are frozen for inference)
catboost_model = train_categorical_model(X_train_cat, y_bin, cat_cols)
incident_model = train_incident_agent(X_train_cat, y_multi, cat_cols)

print("\nSaving Base Models to disk...")
catboost_model.save_model(r"Saves/catboost_base.cbm")
incident_model.save_model(r"Saves/incident_base.cbm")

In [ ]:
# X_train_num is the full-train PCA output from prepare_datasets — correct input
# for the final RCF. X_train_num_raw is NOT used here because the saved PCA
# artifact must be consistent with what predict_proba receives at test time.
rcf_model = RCF(num_trees=40, tree_size=256)
y_bin_reset = y_bin.reset_index(drop=True)
X_train_num_normal_full = X_train_num[y_bin_reset == 0]
rcf_model.fit_predict(X_train_num_normal_full)

rcf_model.save_model(r"Saves/rcf_base.pkl")

In [ ]:
print("\n=======================================================")
print("FINAL HOLD-OUT TEST SET EVALUATION")
print("=======================================================\n")

# Load the OOF-optimised threshold and cost constants from disk so this cell
# runs correctly after a kernel restart without needing cells 2-5 in memory.
import json as _json
_threshold_path = "Saves/optimal_threshold.json"
with open(_threshold_path) as _f:
    _threshold_record = _json.load(_f)
optimal_threshold = _threshold_record["optimal_threshold"]
COST_FN = _threshold_record["cost_fn"]
COST_FP = _threshold_record["cost_fp"]
print(f"Loaded threshold: {optimal_threshold:.4f}  (FN×{COST_FN}, FP×{COST_FP})")

test_file_path = r"UNSW-NB15\unsw_test.csv"

# Test mode: prepare_datasets returns X_num already transformed by the saved
# full-train scaler+PCA. X_num_raw is unused in the test path but unpacked
# for API consistency.
X_test_cat, X_test_num, _, y_test_bin, cat_cols_test, y_test_multi = prepare_datasets(test_file_path, is_train=False)

# Verify PCA dimensionality matches the saved artifact (guards against
# retraining with a different variance threshold or different data)
saved_pca = joblib.load(r"Saves/feature_pca.pkl")
expected_pca_dims = saved_pca.n_components_
assert X_test_num.shape[1] == expected_pca_dims, (
    f"PCA dimensionality mismatch: expected {expected_pca_dims} components, "
    f"got {X_test_num.shape[1]}. Retrain or reload the correct scaler/PCA artifacts."
)

# Load frozen base models
print("\nLoading frozen base models from disk...")
loaded_catboost = CatBoostClassifier()
loaded_catboost.load_model("Saves/catboost_base.cbm")
loaded_rcf = RCF.load_model("Saves/rcf_base.pkl")
loaded_incident = CatBoostClassifier()
loaded_incident.load_model("Saves/incident_base.cbm")

# Generate Level-1 base scores
print("Generating base model scores (CatBoost, RCF, Incident)...")
cat_scores_test = loaded_catboost.predict_proba(X_test_cat)[:, 1]
rcf_scores_test_norm = loaded_rcf.predict_proba(X_test_num)

incident_proba_test = loaded_incident.predict_proba(X_test_cat)
incident_entropy_test = _incident_entropy(incident_proba_test)

# Fuse through meta-learner
print("Fusing scores through the Cost-Sensitive Meta-Learner...")
X_meta_test = np.column_stack((cat_scores_test, rcf_scores_test_norm, incident_entropy_test))
loaded_meta_learner = CostSensitiveMetaLearner.load_model("Saves/meta_learner.pkl")
test_final_risk = loaded_meta_learner.predict_proba(X_meta_test)

# Zero Trust boundary — use the OOF-derived threshold, not 0.5
test_predictions = (test_final_risk >= optimal_threshold).astype(int)

# SOC operational metrics
tn, fp, fn, tp = confusion_matrix(y_test_bin, test_predictions).ravel()
final_soc_cost = (fn * COST_FN) + (fp * COST_FP)

print("\n--- FINAL OPERATIONAL REPORT (TEST SET) ---")
print(f"Decision threshold (OOF-optimised): {optimal_threshold:.4f}")
print(classification_report(y_test_bin, test_predictions, target_names=["Normal (0)", "Attack (1)"]))

print("\n--- ZERO TRUST SOC IMPACT ---")
print(f"True Negatives (Traffic Safely Allowed): {tn}")
print(f"True Positives (Attacks Successfully Blocked): {tp}")
print(f"False Positives (Unjustified Alert Fatigue): {fp}")
print(f"False Negatives (Missed Attacks): {fn}")
print("-------------------------------------------------------")
print(f"Total Operational Penalty Score: {final_soc_cost}")

In [ ]:
# -----------------------------------------------------------------------
# CELL 7 — SOAR / Zero Trust Agent Evaluation
#
# Runs a representative sample of test events through the full
# ZeroTrustSOARAgent pipeline:
#   RCF gate → Policy Agent → Zero Trust diamond → Response Agent (LLM)
#   → SOAR Playbook → Update Trust Score → Feedback Loop
#
# All ML scores come from cell 6 — nothing is recomputed here.
# The agent loads its optimal_threshold from Saves/optimal_threshold.json
# automatically, so this cell is safe to run after a kernel restart.
# -----------------------------------------------------------------------

import json
from soar_zta import ZeroTrustSOARAgent

# --- Build the agent (loads threshold, memory, playbooks, policy config) ---
soar_agent = ZeroTrustSOARAgent()

# --- Select a sample of events to evaluate ---
# Evaluate up to N_SAMPLE events: half predicted attacks, half predicted normal
# so we exercise both branches of the ZeroTrust diamond in one run.
N_SAMPLE = 10
attack_indices = np.where(test_predictions == 1)[0][:N_SAMPLE // 2]
normal_indices = np.where(test_predictions == 0)[0][:N_SAMPLE // 2]
sample_indices = np.concatenate([attack_indices, normal_indices])

# Predicted threat label per test row (from the incident agent in cell 6)
predicted_threat_labels = loaded_incident.predict(X_test_cat)

print("=" * 60)
print("SOAR / ZERO TRUST AGENT — SAMPLE EVENT EVALUATION")
print("=" * 60)

soar_decisions = []

for i, idx in enumerate(sample_indices):
    print(f"\n{'─' * 60}")
    print(f"Event {i + 1}/{len(sample_indices)}  |  Test row index: {idx}")
    print(f"Ground truth: {'ATTACK' if y_test_bin.iloc[idx] == 1 else 'NORMAL'}")

    # Build the context dict from pre-computed cell-6 scores
    telemetry_row = X_test_cat.iloc[idx].to_dict()

    context = soar_agent.construct_context(
        event_id=idx,
        cat_risk=float(cat_scores_test[idx]),
        rcf_risk=float(rcf_scores_test_norm[idx]),
        final_risk=float(test_final_risk[idx]),
        threat_type=str(predicted_threat_labels[idx]),
        telemetry=telemetry_row
    )

    # Run the full pipeline — threshold comes from self.optimal_threshold
    decision = soar_agent.run(event_id=idx, context=context)
    soar_decisions.append(decision)

# --- Summary ---
print(f"\n{'=' * 60}")
print("SAMPLE EVALUATION SUMMARY")
print(f"{'=' * 60}")

playbook_counts = {}
for d in soar_decisions:
    pb = d.get("playbook", "UNKNOWN")
    playbook_counts[pb] = playbook_counts.get(pb, 0) + 1

for playbook, count in sorted(playbook_counts.items()):
    print(f"  {playbook:<22} : {count} event(s)")

fp_overrides = sum(1 for d in soar_decisions if d.get("is_false_positive"))
print(f"\n  LLM False-Positive overrides : {fp_overrides}")
print(f"  Agent memory written to      : {soar_agent.memory_file}")